# Synthetic Data Experiment Series
### All hierarchical projmix cases
- Data generator: hierarchical projected Gaussian
- Data dimension: 2d / 3d
- First-layer separability: high / low
- Regime-directional dependence: strong / weak / independent
- ARI targets: first-layer `ari` and leaf-level `full ari`

This supplements `projmix-all.ipynb`. It uses the same model-comparison loop, but samples from `hier-projmix-configs.json` using a first-layer Gaussian mixture and a conditional second-layer projected-Gaussian mixture.

## 01 - Config and Generator

In [1]:
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path
from time import time

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "cyl_lvm").exists()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cyl_lvm as clvm
from experiment.synthetic_data import experiment_helper as mod

EXPERIMENT_DIR = PROJECT_ROOT / "experiment" / "synthetic_data"
RESULTS_DIR = PROJECT_ROOT / "results" / "synthetic_data"
CONFIG_PATH = EXPERIMENT_DIR / "hier-projmix-configs.json"

with CONFIG_PATH.open("r") as f:
    config_doc = json.load(f)

generator_configs = config_doc["experiments"]
config_doc["configuration_name"], len(generator_configs)

('hier-projmix-configs', 12)

In [2]:
class HierarchicalProjectedGaussianGenerator:
    """Sampler for one case in hier-projmix-configs.json.

    Returned labels contain first- and second-layer assignments in sample order.
    """

    def __init__(self, cfg):
        self.cfg = cfg
        self.d_gauss = int(cfg["dimensions"]["linear"])
        self.d_vmf = int(cfg["dimensions"]["projected_latent"])

        self.weights = np.asarray(cfg["first_layer"]["mixture_weights"], dtype=float)
        self.layer1_means = np.asarray(cfg["first_layer"]["component_means"], dtype=float)
        self.layer1_covariances = np.asarray(cfg["first_layer"]["covariances"], dtype=float)

        self.conditional_weights = np.asarray(
            cfg["second_layer"]["conditional_mixture_weights"],
            dtype=float,
        )
        self.layer2_means = np.asarray(
            cfg["second_layer"]["pre_projection_means"],
            dtype=float,
        )
        self.layer2_covariances = np.asarray(
            cfg["second_layer"]["pre_projection_covariances"],
            dtype=float,
        )

        self.n_first_layer = self.weights.size
        self.n_second_layer = self.conditional_weights.shape[1]
        self._validate()

    def _validate(self):
        K = self.n_first_layer
        L = self.n_second_layer
        dg = self.d_gauss
        dv = self.d_vmf

        if self.layer1_means.shape != (K, dg):
            raise ValueError(f"first-layer mean shape mismatch: {self.layer1_means.shape}")
        if self.layer1_covariances.shape != (K, dg, dg):
            raise ValueError(f"first-layer covariance shape mismatch: {self.layer1_covariances.shape}")
        if self.conditional_weights.shape != (K, L):
            raise ValueError(f"conditional weight shape mismatch: {self.conditional_weights.shape}")
        if self.layer2_means.shape != (K, L, dv):
            raise ValueError(f"second-layer mean shape mismatch: {self.layer2_means.shape}")
        if self.layer2_covariances.shape != (K, L, dv, dv):
            raise ValueError(f"second-layer covariance shape mismatch: {self.layer2_covariances.shape}")
        if not np.isclose(self.weights.sum(), 1.0):
            raise ValueError("first-layer weights must sum to 1")
        if not np.allclose(self.conditional_weights.sum(axis=1), 1.0):
            raise ValueError("each conditional weight row must sum to 1")

    def sample(self, n, *, rng=None, return_labels=False):
        if rng is None:
            rng = np.random.RandomState()

        hidden_y1 = rng.choice(self.n_first_layer, size=n, p=self.weights)
        x_gauss = np.empty((n, self.d_gauss), dtype=float)
        x_vmf = np.empty((n, self.d_vmf), dtype=float)
        hidden_y2 = np.empty(n, dtype=int)

        for j in range(self.n_first_layer):
            idx_j = np.flatnonzero(hidden_y1 == j)
            if idx_j.size == 0:
                continue

            x_gauss[idx_j] = rng.multivariate_normal(
                self.layer1_means[j],
                self.layer1_covariances[j],
                size=idx_j.size,
            )

            hidden_y2_j = rng.choice(
                self.n_second_layer,
                size=idx_j.size,
                p=self.conditional_weights[j],
            )
            hidden_y2[idx_j] = hidden_y2_j
            for k in range(self.n_second_layer):
                local_idx = np.flatnonzero(hidden_y2_j == k)
                if local_idx.size == 0:
                    continue

                idx_jk = idx_j[local_idx]
                latent_direction = rng.multivariate_normal(
                    self.layer2_means[j, k],
                    self.layer2_covariances[j, k],
                    size=idx_jk.size,
                )
                x_vmf[idx_jk] = mod.unit(latent_direction)

        x = np.concatenate((x_gauss, x_vmf), axis=1)
        if return_labels:
            return x, np.stack([hidden_y1, hidden_y2], axis=1)
        return x


def build_generator(cfg):
    return HierarchicalProjectedGaussianGenerator(cfg)


def first_layer_labels(labels):
    """Extract first-layer regime labels for the regular ARI target."""
    labels = np.asarray(labels)
    if labels.ndim == 1:
        return labels
    return labels[:, 0]


def full_layer_labels(labels):
    """Convert (Y1, Y2) pairs into one compact leaf-cluster label."""
    labels = np.asarray(labels)
    if labels.ndim != 2 or labels.shape[1] != 2:
        raise ValueError("full-layer labels must have shape (n_samples, 2).")
    _, encoded = np.unique(labels, axis=0, return_inverse=True)
    return encoded

In [3]:
# Fast structural smoke check before launching the full experiment.
for key, cfg in generator_configs.items():
    rng = np.random.RandomState(20260722)
    generator = build_generator(cfg)
    x, labels = generator.sample(256, rng=rng, return_labels=True)
    d_gauss = cfg["dimensions"]["linear"]
    assert x.shape == (256, 2 * d_gauss), key
    assert labels.shape == (256, 2), key
    assert first_layer_labels(labels).shape == (256,), key
    assert full_layer_labels(labels).shape == (256,), key
    assert np.allclose(np.linalg.norm(x[:, d_gauss:], axis=1), 1.0), key

pd.DataFrame(
    [
        {
            "Experiment": key.replace("hier-projmix-", "", 1),
            "Dimension": cfg["dimensions"]["linear"],
            "Separability": cfg["factor_levels"]["component_separability"],
            "Dependence": cfg["factor_levels"]["regime_directional_dependence"],
            "K": cfg["latent_structure"]["n_first_layer_components"],
            "L": cfg["latent_structure"]["n_second_layer_components_per_regime"],
        }
        for key, cfg in generator_configs.items()
    ]
)

,Experiment,Dimension,Separability,Dependence,K,L
0,2d-low-strong,2,low,strong,3,2
1,2d-low-weak,2,low,weak,3,2
2,2d-low-ind,2,low,independent,3,2
3,2d-high-strong,2,high,strong,3,2
4,2d-high-weak,2,high,weak,3,2
5,2d-high-ind,2,high,independent,3,2
6,3d-low-strong,3,low,strong,3,2
7,3d-low-weak,3,low,weak,3,2
8,3d-low-ind,3,low,independent,3,2
9,3d-high-strong,3,high,strong,3,2


## 02 - Model Grid

In [4]:
N = 10000
N_train = int(N * 0.8)
NOISE_SCALE = 0.15
n_seeds = 100
seeds = range(n_seeds)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_AGG_PATH = RESULTS_DIR / "hier-projmix_results_raw.csv"
OUTPUT_LONG_PATH = RESULTS_DIR / "hier-projmix_results_raw_long.csv"

setup_list = [
    {"model_type": "cylmix", "model_components": [2]},
    {"model_type": "indcylmix", "model_components": [2]},
    {"model_type": "mom", "model_components": [2, 1]},
    {"model_type": "mom", "model_components": [2, 2]},
    {"model_type": "mom", "model_components": [2, 3]},
    {"model_type": "isomom", "model_components": [2, 1]},
    {"model_type": "isomom", "model_components": [2, 2]},
    {"model_type": "isomom", "model_components": [2, 3]},

    {"model_type": "cylmix", "model_components": [3]},
    {"model_type": "indcylmix", "model_components": [3]},
    {"model_type": "mom", "model_components": [3, 1]},
    {"model_type": "mom", "model_components": [3, 2]},
    {"model_type": "mom", "model_components": [3, 3]},
    {"model_type": "isomom", "model_components": [3, 1]},
    {"model_type": "isomom", "model_components": [3, 2]},
    {"model_type": "isomom", "model_components": [3, 3]},

    {"model_type": "cylmix", "model_components": [4]},
    {"model_type": "indcylmix", "model_components": [4]},
    {"model_type": "mom", "model_components": [4, 1]},
    {"model_type": "mom", "model_components": [4, 2]},
    {"model_type": "mom", "model_components": [4, 3]},
    {"model_type": "isomom", "model_components": [4, 1]},
    {"model_type": "isomom", "model_components": [4, 2]},
    {"model_type": "isomom", "model_components": [4, 3]},

    {"model_type": "cylmix", "model_components": [5]},
    {"model_type": "indcylmix", "model_components": [5]},

    {"model_type": "cylmix", "model_components": [6]},
    {"model_type": "indcylmix", "model_components": [6]},
]

model_names = [
    "2-Cylindrical Mixture",
    "2-Independent Cylindrical Mixture",
    "(2,1)-Two-layer MoM",
    "(2,2)-Two-layer MoM",
    "(2,3)-Two-layer MoM",
    "(2,1)-Isolated Two-layer MoM",
    "(2,2)-Isolated Two-layer MoM",
    "(2,3)-Isolated Two-layer MoM",

    "3-Cylindrical Mixture",
    "3-Independent Cylindrical Mixture",
    "(3,1)-Two-layer MoM",
    "(3,2)-Two-layer MoM",
    "(3,3)-Two-layer MoM",
    "(3,1)-Isolated Two-layer MoM",
    "(3,2)-Isolated Two-layer MoM",
    "(3,3)-Isolated Two-layer MoM",

    "4-Cylindrical Mixture",
    "4-Independent Cylindrical Mixture",
    "(4,1)-Two-layer MoM",
    "(4,2)-Two-layer MoM",
    "(4,3)-Two-layer MoM",
    "(4,1)-Isolated Two-layer MoM",
    "(4,2)-Isolated Two-layer MoM",
    "(4,3)-Isolated Two-layer MoM",

    "5-Cylindrical Mixture",
    "5-Independent Cylindrical Mixture",

    "6-Cylindrical Mixture",
    "6-Independent Cylindrical Mixture",
]

experiment_order = [key.replace("hier-projmix-", "", 1) for key in generator_configs.keys()]

## 03 - Run Experiments

In [5]:
records = []
total_progress = len(generator_configs) * len(seeds)
progress_idx = 0

for key, cfg in generator_configs.items():
    case = key.replace("hier-projmix-", "", 1)
    case_start = time()

    d_gauss = cfg["dimensions"]["linear"]
    d_vmf = cfg["dimensions"]["projected_latent"]
    generator = build_generator(cfg)

    for seed in seeds:
        seed_start = time()
        rng = np.random.RandomState(seed)

        sample = mod.sample_noisy_train_test(
            generator,
            n=N,
            n_train=N_train,
            d_gauss=d_gauss,
            d_vmf=d_vmf,
            rng=rng,
            noise_scale=NOISE_SCALE,
        )

        labels_train = first_layer_labels(sample["labels_train"])
        labels_test = first_layer_labels(sample["labels_test"])
        full_labels_train = full_layer_labels(sample["labels_train"])
        full_labels_test = full_layer_labels(sample["labels_test"])

        models, training_times, n_iters = mod.train_all_models(
            d_gauss,
            sample["x_train"],
            setup_list=setup_list,
            print_=False,
            return_training_times=True,
            return_em_iter=True,
        )
        models_with_noise, training_times_with_noise, n_iters_with_noise = mod.train_all_models(
            d_gauss,
            sample["x_train_noise"],
            setup_list=setup_list,
            print_=False,
            return_training_times=True,
            return_em_iter=True,
        )

        runs = {
            "no noise": {
                "models": models,
                "training_times": training_times,
                "em_iters": n_iters,
                "x_train": sample["x_train"],
                "x_test": sample["x_test"],
            },
            "with noise": {
                "models": models_with_noise,
                "training_times": training_times_with_noise,
                "em_iters": n_iters_with_noise,
                "x_train": sample["x_train_noise"],
                "x_test": sample["x_test_noise"],
            },
        }

        for noise, run in runs.items():
            records.append({
                "Seed": seed,
                "Model": "all",
                "Model Type": "all",
                "Metric": "sample_cross_cov",
                "Sample": "training",
                "Noise": noise,
                "Experiment": case,
                "Value": round(
                    np.linalg.norm(
                        np.cov(run["x_train"], rowvar=False)[:d_gauss, d_gauss:],
                        ord="fro",
                    ),
                    2,
                ),
            })

            for model_name, setup, elapsed, em_iters in zip(
                model_names,
                setup_list,
                run["training_times"],
                run["em_iters"],
            ):
                records.append({
                    "Seed": seed,
                    "Model": model_name,
                    "Model Type": setup["model_type"],
                    "Metric": "training_time",
                    "Sample": "training",
                    "Noise": noise,
                    "Experiment": case,
                    "Value": round(elapsed * 1000, 2),
                })
                records.append({
                    "Seed": seed,
                    "Model": model_name,
                    "Model Type": setup["model_type"],
                    "Metric": "em_iters",
                    "Sample": "training",
                    "Noise": noise,
                    "Experiment": case,
                    "Value": int(em_iters),
                })

            for sample_name, x_eval, labels, full_labels in [
                ("in sample", run["x_train"], labels_train, full_labels_train),
                ("out of sample", run["x_test"], labels_test, full_labels_test),
            ]:
                for model, model_name, setup in zip(run["models"], model_names, setup_list):
                    records.append({
                        "Seed": seed,
                        "Model": model_name,
                        "Model Type": setup["model_type"],
                        "Metric": "ari",
                        "Sample": sample_name,
                        "Noise": noise,
                        "Experiment": case,
                        "Value": round(mod.ari_model(model,
                                                     labels,
                                                     x_eval,
                                                     d_gauss), 5),
                    })

                    if hasattr(model, "predict_full"):
                        records.append({
                            "Seed": seed,
                            "Model": model_name,
                            "Model Type": setup["model_type"],
                            "Metric": "full ari",
                            "Sample": sample_name,
                            "Noise": noise,
                            "Experiment": case,
                            "Value": round(mod.ari_full_model(model,
                                                         full_labels,
                                                         x_eval,
                                                         d_gauss), 5),
                        })

                    for score in ["gmpd", "avg_ll", "bic", "aic"]:
                        records.append({
                            "Seed": seed,
                            "Model": model_name,
                            "Model Type": setup["model_type"],
                            "Metric": score,
                            "Sample": sample_name,
                            "Noise": noise,
                            "Experiment": case,
                            "Value": round(
                                mod.score_model(model, x_eval, d_gauss, score_type=score),
                                5,
                            ),
                        })

        progress_idx += 1
        print(
            f"{progress_idx / total_progress * 100:.2f}% | "
            f"{case} | seed={seed} | "
            f"seed time: {time() - seed_start:.0f}s | "
            f"case elapsed: {time() - case_start:.0f}s",
            flush=True,
        )

    print(
        f"{progress_idx}/{total_progress} | "
        f"{case} complete | total case time: {time() - case_start:.0f}s",
        flush=True,
    )

results_long = pd.DataFrame(records)
results_long.head()

0.08% | 2d-low-strong | seed=0 | seed time: 56s | case elapsed: 56s
0.17% | 2d-low-strong | seed=1 | seed time: 48s | case elapsed: 104s
0.25% | 2d-low-strong | seed=2 | seed time: 42s | case elapsed: 146s
0.33% | 2d-low-strong | seed=3 | seed time: 42s | case elapsed: 189s
0.42% | 2d-low-strong | seed=4 | seed time: 43s | case elapsed: 232s
0.50% | 2d-low-strong | seed=5 | seed time: 43s | case elapsed: 274s
0.58% | 2d-low-strong | seed=6 | seed time: 44s | case elapsed: 318s
0.67% | 2d-low-strong | seed=7 | seed time: 44s | case elapsed: 362s
0.75% | 2d-low-strong | seed=8 | seed time: 47s | case elapsed: 408s
0.83% | 2d-low-strong | seed=9 | seed time: 45s | case elapsed: 453s
0.92% | 2d-low-strong | seed=10 | seed time: 43s | case elapsed: 496s
1.00% | 2d-low-strong | seed=11 | seed time: 44s | case elapsed: 540s
1.08% | 2d-low-strong | seed=12 | seed time: 44s | case elapsed: 584s
1.17% | 2d-low-strong | seed=13 | seed time: 44s | case elapsed: 627s
1.25% | 2d-low-strong | seed=14

,Seed,Model,Model Type,Metric,Sample,Noise,Experiment,Value
0,0,all,all,sample_cross_cov,training,no noise,2d-low-strong,0.04
1,0,2-Cylindrical Mixture,cylmix,training_time,training,no noise,2d-low-strong,444.68
2,0,2-Cylindrical Mixture,cylmix,em_iters,training,no noise,2d-low-strong,17.00
3,0,2-Independent Cylindrical Mixture,indcylmix,training_time,training,no noise,2d-low-strong,408.65
4,0,2-Independent Cylindrical Mixture,indcylmix,em_iters,training,no noise,2d-low-strong,18.00


## 04 - Aggregate and Save

In [6]:
experiment_parts = results_long["Experiment"].str.extract(
    r"^(?P<Dimension>\d+)d-(?P<Separability>[^-]+)-(?P<Dependence>[^-]+)$"
)

results_long = results_long.assign(
    Dimension=experiment_parts["Dimension"].astype(int),
    Separability=experiment_parts["Separability"],
    Dependence=experiment_parts["Dependence"],
)

In [7]:
results_long[["K", "L"]] = results_long["Model"].str.extract(
    r"^\(?(\d+)(?:,(\d+))?"
)
results_long["K"] = results_long["K"].fillna(0).astype(int)
results_long["L"] = results_long["L"].fillna(0).astype(int)

results_long.columns

Index(['Seed', 'Model', 'Model Type', 'Metric', 'Sample', 'Noise',
       'Experiment', 'Value', 'Dimension', 'Separability', 'Dependence', 'K',
       'L'],
      dtype='object')

In [8]:
agg_cols = [
    "Model",
    "Model Type",
    "Metric",
    "Sample",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "K",
    "L",
]

results_agg = (
    results_long
    .groupby(agg_cols, as_index=False)
    .agg(
        **{
            "Avg Value": ("Value", "mean"),
            "Std Value": ("Value", "std"),
        }
    )
)

results_agg["Std Value"] = results_agg["Std Value"].fillna(0.0)
results_agg.head()

,Model,Model Type,Metric,Sample,Noise,Experiment,Dimension,Separability,Dependence,K,L,Avg Value,Std Value
0,"(2,1)-Isolated Two-layer MoM",isomom,aic,in sample,no noise,2d-high-ind,2,high,ind,2,1,64721.638438,211.779198
1,"(2,1)-Isolated Two-layer MoM",isomom,aic,in sample,no noise,2d-high-strong,2,high,strong,2,1,71262.636101,204.201812
2,"(2,1)-Isolated Two-layer MoM",isomom,aic,in sample,no noise,2d-high-weak,2,high,weak,2,1,65735.387111,210.308129
3,"(2,1)-Isolated Two-layer MoM",isomom,aic,in sample,no noise,2d-low-ind,2,low,ind,2,1,54238.682179,242.288861
4,"(2,1)-Isolated Two-layer MoM",isomom,aic,in sample,no noise,2d-low-strong,2,low,strong,2,1,57295.303607,230.967491


In [9]:
results_agg.to_csv(
    OUTPUT_AGG_PATH,
    index=False
)

OUTPUT_AGG_PATH

PosixPath('/Users/jgv/PycharmProjects/cylindrical-data-lvm/results/synthetic_data/hier-projmix_results_raw.csv')

In [10]:
results_long.to_csv(
    OUTPUT_LONG_PATH,
    index=False
)

OUTPUT_LONG_PATH

PosixPath('/Users/jgv/PycharmProjects/cylindrical-data-lvm/results/synthetic_data/hier-projmix_results_raw_long.csv')